In [1]:
# Ollama default port: 11434
# ssh -L 11434:localhost:11434 USERNAME@HOST

### Imports

In [2]:
from testgen.utils import *
import pandas as pd
import json
import os
from tqdm.notebook import tqdm
from pathlib import Path
from datetime import datetime as dt
from dotenv import load_dotenv

In [3]:
base_path = Path().cwd()
data_path = base_path / "data"
load_dotenv(override=True)

True

In [4]:
# read data and rename columns
sensor_df = pd.read_excel(data_path / "sensor_requirements.xlsx")
sensor_df_examples = pd.read_excel(data_path / "sensor_examples.xlsx")

actuator_df = pd.read_excel(data_path / "actuator_requirements.xlsx")
actuator_df_examples = pd.read_excel(data_path / "actuator_examples.xlsx")

In [5]:
sensor_df.columns = list(map(lambda x: x.lower().strip(), sensor_df.columns))
sensor_df_examples.columns = list(map(lambda x: x.lower().strip(), sensor_df_examples.columns))
actuator_df.columns = list(map(lambda x: x.lower().strip(), actuator_df.columns))
actuator_df_examples.columns = list(map(lambda x: x.lower().strip(), actuator_df_examples.columns))

In [6]:
sensor_df["target_actuator"] = 0
sensor_df_examples["target_actuator"] = 0
actuator_df["target_actuator"] = 1
actuator_df_examples["target_actuator"] = 1

In [7]:
sensor_df = sensor_df[["requirement", "target_actuator"]]
sensor_df_examples = sensor_df_examples[["requirement", "target_actuator"]]
actuator_df = actuator_df[["requirement", "target_actuator"]]
actuator_df_examples = actuator_df_examples[["requirement", "target_actuator"]]

# Format Response

In [8]:
from pydantic import BaseModel, Field

class TargetActuator(BaseModel):
    target_actuator: int = Field(description="Target actuator (0 for sensor, 1 for actuator)")

# Formulate Examples

In [9]:
def generate_examples(N_EXAMPLES):
    example_template = """<Example {i}>
    text: {text}
    target_actuator: {target_actuator}
    </Example {i}>"""

    examples = sensor_df_examples.sample(N_EXAMPLES).copy().to_dict(orient="records")
    examples.extend(
        actuator_df_examples.sample(N_EXAMPLES).copy().to_dict(orient="records")
    )

    examples_text = [
        example_template.format(
            i=i, text=e["requirement"], target_actuator=e["target_actuator"]
        )
        for i, e in enumerate(examples, start=1)
    ]
    examples_text = "\n".join(examples_text)

    return examples, examples_text

# LLM

In [10]:
from testgen.prompts.SensorActuator import SensorActuator

In [11]:
llm_models = {
    "ollama": ["phi4:latest"],
}

endpoint_attrs = {
    "ollama": {
        "api_key": os.getenv("OLLAMA_API_KEY"),
        "base_url": os.getenv("OLLAMA_ENDPOINT"),
    }
}

# Utils for all Requirements

In [12]:
# combine both dataframes and shuffle
df = (
    pd.concat([sensor_df, actuator_df], ignore_index=True)
    .sample(frac=1)
    .reset_index(drop=True)
)

# Run against all models

In [13]:
for N_EXAMPLES in [1, 3, 5]:

    examples, examples_text = generate_examples(N_EXAMPLES)

    for endpoint_name in llm_models.keys():

        for model_name in llm_models[endpoint_name]:

            print(f"Running {model_name} on {endpoint_name} with n {N_EXAMPLES} ...")

            client = llm_client(endpoint_name, **endpoint_attrs[endpoint_name])

            results = client_invoke_actuator_sensor(
                endpoint_name,
                client,
                df,
                model_name,
                SensorActuator,
                examples_text,
                response_format=TargetActuator,
            )

            (
                number_of_requests,
                accuracy,
                avg_time_per_req,
                avg_token_per_req,
                avg_completion_token_per_req,
                total_tokens,
                total_completion_tokens,
                total_time,
            ) = calc_stats(results)

            results_file = save_responses(
                base_path,
                "sensor-actuator_single",
                model_name=model_name.split("/")[-1],
                n_examples=N_EXAMPLES,
                examples=examples,
                accuracy=accuracy,
                number_of_requests=number_of_requests,
                total_tokens=total_tokens,
                total_completion_tokens=total_completion_tokens,
                avg_token_per_req=avg_token_per_req,
                avg_completion_token_per_req=avg_completion_token_per_req,
                avg_time_per_req=avg_time_per_req,
                results=results,
            )

            print(f"Done")

Running phi4:latest on ollama with n 1 ...


  0%|          | 0/134 [00:00<?, ?it/s]

Done
Running phi4:latest on ollama with n 3 ...


  0%|          | 0/134 [00:00<?, ?it/s]

Done
Running phi4:latest on ollama with n 5 ...


  0%|          | 0/134 [00:00<?, ?it/s]

Done
